In [2]:
import re
import string
import numpy as np
import pandas as pd
from collections import Counter
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

pd.set_option('display.max_colwidth', None)

## Load Datasets

In [2]:
train = pd.read_parquet('/home/inductive_anks/Machine Learning/Competitions/Zindi/Machine Translation/Machine Translation/data/raw/train-00000-of-00001.parquet')

In [3]:
test = pd.read_parquet('/home/inductive_anks/Machine Learning/Competitions/Zindi/Machine Translation/Machine Translation/data/raw/test-00000-of-00001.parquet')

In [4]:
validate = pd.read_parquet('/home/inductive_anks/Machine Learning/Competitions/Zindi/Machine Translation/Machine Translation/data/raw/validation-00000-of-00001.parquet')

In [5]:
train.head()

,ID,translation
0,ID_18897661270129,"{'dyu': 'A bi ji min na', 'fr': 'Il boit de l’eau.'}"
1,ID_18479132727846,"{'dyu': 'A le dalakolontɛ lon bɛ.', 'fr': 'Il se plaint toujours.'}"
2,ID_18164131280307,"{'dyu': 'Mun? Fɛn dɔ.', 'fr': 'Quoi ? Quelque chose.'}"
3,ID_18344573728152,"{'dyu': 'O bɛ bi bɔra fo Gubeta.', 'fr': 'Tous sortent excepté Gubetta.'}"
4,ID_18127342282717,"{'dyu': 'A ale lo bi da bugɔ la!', 'fr': 'Ah ! c’est lui… il sonne…'}"


In [6]:
test.head()

,ID,translation
0,ID_17345911362699,"{'dyu': 'An kelen duron le tun be yi', 'fr': '0'}"
1,ID_173626847.3381,"{'dyu': 'O ka papiye farana.', 'fr': '0'}"
2,ID_17704632382547,"{'dyu': 'N tɔrɔla kɔ tuguni', 'fr': '0'}"
3,ID_19793499384156,"{'dyu': 'I tun b'a daminɛ tan kɛ.', 'fr': '0'}"
4,ID_17802727385575,"{'dyu': 'A kɛra ka ban.', 'fr': '0'}"


In [7]:
validate.head()

,ID,translation
0,ID_17914990255818,"{'dyu': 'I tɔgɔ bi cogodɔ', 'fr': 'Tu portes un nom de fantaisie.'}"
1,ID_18135961264225,"{'dyu': 'Puɛn saba fɔlɔ.', 'fr': 'Trois points d’avance.'}"
2,ID_18161475265686,"{'dyu': 'Tile bena', 'fr': 'Le soleil s’est couché.'}"
3,ID_17305345266745,"{'dyu': 'cogoya kelen', 'fr': 'Mêmes mouvements.'}"
4,ID_18106593267767,"{'dyu': 'N ma daraka dun ban.', 'fr': 'Je n’ai pas encore déjeuné.'}"


In [8]:
print('Length of train data: ', len(train))
print('Length of test data: ', len(test))
print('Length of validate data: ', len(validate))

Length of train data:  8065
Length of test data:  1393
Length of validate data:  1471


## Finding Punctuations in the Data

In [9]:
train['dyu'] = train['translation'].apply(lambda x: x['dyu'])
train['fr'] = train['translation'].apply(lambda x: x['fr'])

test['dyu'] = train['translation'].apply(lambda x: x['dyu'])
test['fr'] = train['translation'].apply(lambda x: x['fr'])

validate['dyu'] = train['translation'].apply(lambda x: x['dyu'])
validate['fr'] = train['translation'].apply(lambda x: x['fr'])


In [10]:
def find_punctuation(text):
    punctuation = re.findall(r'[^\w\s]', text)
    return punctuation

In [11]:
train['dyu_punctuation'] = train['dyu'].apply(find_punctuation)
train['fr_punctuation'] = train['fr'].apply(find_punctuation)

validate['dyu_punctuation'] = validate['dyu'].apply(find_punctuation)
validate['fr_punctuation'] = validate['fr'].apply(find_punctuation)

test['dyu_punctuation'] = test['dyu'].apply(find_punctuation)

train_dyu_punctuation_count = Counter([p for sublist in train['dyu_punctuation'] for p in sublist])
train_fr_punctuation_count = Counter([p for sublist in train['fr_punctuation'] for p in sublist])

validate_dyu_punctuation_count = Counter([p for sublist in validate['dyu_punctuation'] for p in sublist])
validate_fr_punctuation_count = Counter([p for sublist in validate['fr_punctuation'] for p in sublist])

test_dyu_punctuation_count = Counter([p for sublist in test['dyu_punctuation'] for p in sublist])


In [12]:
print("Train Dyula Punctuation Count:", train_dyu_punctuation_count)
print("\nTrain French Punctuation Count:", train_fr_punctuation_count)

print("\nValidate Dyula Punctuation Count:", validate_dyu_punctuation_count)
print("\nValidate French Punctuation Count:", validate_fr_punctuation_count)

print("\nTest Dyula Punctuation Count:", test_dyu_punctuation_count)

Train Dyula Punctuation Count: Counter({'.': 2247, "'": 1037, '?': 540, ',': 210, '!': 63, '-': 27, '́': 2, ';': 1, '[': 1, '/': 1})

Train French Punctuation Count: Counter({'.': 5825, ',': 1774, "'": 1558, '’': 1378, '-': 1331, '?': 1007, '!': 645, '…': 301, '́': 75, '̀': 42, '̂': 31, ':': 21, '—': 20, '«': 12, ';': 11, '»': 11, '/': 9, '(': 7, ')': 7, '=': 6, '̧': 5, '"': 3, '“': 3, '”': 3, '–': 2, '&': 1, '€': 1})

Validate Dyula Punctuation Count: Counter({'.': 383, "'": 180, '?': 119, ',': 19, '!': 16, '́': 2, '-': 1})

Validate French Punctuation Count: Counter({'.': 1079, "'": 243, '’': 239, ',': 220, '-': 214, '?': 206, '!': 117, '…': 54, '́': 11, '̀': 4, '=': 4, '̂': 3, ')': 2, '»': 2, ';': 1, '(': 1, '–': 1, '"': 1, ':': 1})

Test Dyula Punctuation Count: Counter({'.': 371, "'": 178, '?': 111, ',': 18, '!': 16, '́': 2, '-': 1})


# Removing Punctuations

In [13]:
def remove_punctuation(text):
    # Ensure text is a string
    if isinstance(text, str):
        translator = str.maketrans('', '', string.punctuation)
        return text.translate(translator)
    else:
        return text

In [14]:
train['dyu_no_punctuation'] = train['dyu'].apply(remove_punctuation)
train['fr_no_punctuation'] = train['fr'].apply(remove_punctuation)

validate['dyu_no_punctuation'] = validate['dyu'].apply(remove_punctuation)
validate['fr_no_punctuation'] = validate['fr'].apply(remove_punctuation)

test['dyu_no_punctuation'] = test['dyu'].apply(remove_punctuation)

In [15]:
train_dyu_punctuation_count = Counter([p for sublist in train['dyu_no_punctuation'] for p in sublist])
train_fr_punctuation_count = Counter([p for sublist in train['fr_no_punctuation'] for p in sublist])

validate_dyu_punctuation_count = Counter([p for sublist in validate['dyu_no_punctuation'] for p in sublist])
validate_fr_punctuation_count = Counter([p for sublist in validate['fr_no_punctuation'] for p in sublist])

test_dyu_punctuation_count = Counter([p for sublist in test['dyu_no_punctuation'] for p in sublist])

In [16]:
print("Train Dyula Punctuation Count:", train_dyu_punctuation_count)
print("\nTrain French Punctuation Count:", train_fr_punctuation_count)

print("\nValidate Dyula Punctuation Count:", validate_dyu_punctuation_count)
print("\nValidate French Punctuation Count:", validate_fr_punctuation_count)

print("\nTest Dyula Punctuation Count:", test_dyu_punctuation_count)

Train Dyula Punctuation Count: Counter({' ': 40144, 'a': 25362, 'n': 17555, 'i': 15341, 'k': 8012, 'o': 7421, 'l': 7306, 'e': 6790, 'r': 6334, 'b': 5956, 'ɛ': 5862, 'u': 5834, 'ɔ': 5451, 'm': 5103, 't': 4528, 's': 4522, 'y': 4400, 'g': 3632, 'd': 2978, 'f': 2610, 'A': 2092, 'w': 2062, 'è': 1186, 'j': 980, 'N': 923, 'ɲ': 898, 'c': 897, 'é': 648, 'K': 632, 'p': 543, 'ô': 541, 'S': 501, 'h': 472, 'M': 468, 'O': 401, 'I': 384, 'B': 354, 'v': 302, 'D': 265, 'z': 260, 'T': 230, 'F': 220, 'L': 183, 'P': 169, 'â': 158, 'J': 138, 'Y': 118, 'ê': 106, 'G': 105, 'C': 91, 'W': 91, 'R': 91, 'E': 76, 'H': 62, 'V': 56, 'Z': 50, 'U': 34, 'Ɔ': 28, 'Ɛ': 20, 'q': 20, 'x': 17, 'à': 16, 'Ɲ': 12, 'ŋ': 8, 'ç': 5, 'É': 5, 'Q': 3, '́': 2, 'ë': 2, '0': 2, 'î': 2, 'ñ': 2, 'ï': 1, '1': 1, 'Ô': 1, 'û': 1, 'ò': 1, 'À': 1, 'ã': 1, 'ó': 1, 'ā': 1})

Train French Punctuation Count: Counter({' ': 37839, 'e': 30170, 's': 16305, 'a': 15822, 'i': 14477, 'n': 14285, 't': 14200, 'r': 13131, 'u': 12844, 'l': 11066, 'o': 11034

In [17]:
test.head(1)

,ID,translation,dyu,fr,dyu_punctuation,dyu_no_punctuation
0,ID_17345911362699,"{'dyu': 'An kelen duron le tun be yi', 'fr': '0'}",A bi ji min na,Il boit de l’eau.,[],A bi ji min na


In [18]:
train.drop(columns=['translation', 'dyu', 'fr', 'dyu_punctuation', 'fr_punctuation'], inplace=True)
validate.drop(columns=['translation', 'dyu', 'fr', 'dyu_punctuation', 'fr_punctuation'], inplace=True)
test.drop(columns=['translation', 'dyu', 'fr', 'dyu_punctuation'], inplace=True)

In [19]:
train.rename(columns={'dyu_no_punctuation': 'dyu', 'fr_no_punctuation': 'fr'}, inplace=True)
validate.rename(columns={'dyu_no_punctuation': 'dyu', 'fr_no_punctuation': 'fr'}, inplace=True)

In [20]:
test.rename(columns={'dyu_no_punctuation': 'dyu'}, inplace=True)

In [21]:
validate.head()

,ID,dyu,fr
0,ID_17914990255818,A bi ji min na,Il boit de l’eau
1,ID_18135961264225,A le dalakolontɛ lon bɛ,Il se plaint toujours
2,ID_18161475265686,Mun Fɛn dɔ,Quoi Quelque chose
3,ID_17305345266745,O bɛ bi bɔra fo Gubeta,Tous sortent excepté Gubetta
4,ID_18106593267767,A ale lo bi da bugɔ la,Ah c’est lui… il sonne…


## Converting text to Lower Case

In [22]:
train['dyu'] = train['dyu'].str.lower()
train['fr'] = train['fr'].str.lower()

validate['dyu'] = validate['dyu'].str.lower()
validate['fr'] = validate['fr'].str.lower()

test['dyu'] = test['dyu'].str.lower()

## Creating Word Corpus

In [23]:
train.head()

,ID,dyu,fr
0,ID_18897661270129,a bi ji min na,il boit de l’eau
1,ID_18479132727846,a le dalakolontɛ lon bɛ,il se plaint toujours
2,ID_18164131280307,mun fɛn dɔ,quoi quelque chose
3,ID_18344573728152,o bɛ bi bɔra fo gubeta,tous sortent excepté gubetta
4,ID_18127342282717,a ale lo bi da bugɔ la,ah c’est lui… il sonne…


In [24]:
train_dyu_corpus = train['dyu'].tolist()
train_fr_corpus = train['fr'].tolist()

validate_dyu_corpus = validate['dyu'].tolist()
validate_fr_corpus = validate['fr'].tolist()

test_dyu_corpus = test['dyu'].tolist()

## Tokenization of the Sentences in Words

In [25]:
train_dyu_unique_words = set(word for sentence in train_dyu_corpus for word in sentence.split())
train_fr_unique_words = set(word for sentence in train_fr_corpus for word in sentence.split())

validate_dyu_unique_words = set(word for sentence in validate_dyu_corpus for word in sentence.split())
validate_fr_unique_words = set(word for sentence in validate_fr_corpus for word in sentence.split())

test_dyu_unique_words = set(word for sentence in test_dyu_corpus for word in sentence.split())

In [26]:
num_unique_dyu_words = len(train_dyu_unique_words)
num_unique_fr_words = len(train_fr_unique_words)

In [27]:
print('Total No. of Unique Words in dyu Train Vocab: ', num_unique_dyu_words)
print('Total No. of Unique Words in fr Train Vocab: ', num_unique_fr_words)

Total No. of Unique Words in dyu Train Vocab:  7732
Total No. of Unique Words in fr Train Vocab:  10066


## Converting Words into numbers

In [28]:
dyu_corpus = train['dyu'].tolist() + validate['dyu'].tolist() + test['dyu'].tolist()
fr_corpus = train['fr'].tolist() + validate['fr'].tolist()

In [29]:
dyu_tokenizer = Tokenizer()
dyu_tokenizer.fit_on_texts(dyu_corpus)

fr_tokenizer = Tokenizer()
fr_tokenizer.fit_on_texts(fr_corpus)

In [30]:
train_dyu_sequences = dyu_tokenizer.texts_to_sequences(train['dyu'].tolist())
train_fr_sequences = fr_tokenizer.texts_to_sequences(train['fr'].tolist())

validate_dyu_sequences = dyu_tokenizer.texts_to_sequences(validate['dyu'].tolist())
validate_fr_sequences = fr_tokenizer.texts_to_sequences(validate['fr'].tolist())

In [31]:
test_dyu_sequences = dyu_tokenizer.texts_to_sequences(test['dyu'].tolist())

In [32]:
print("First 5 Dyula sequences in training data:", train_dyu_sequences[:5])
print("First 5 French sequences in training data:", train_fr_sequences[:5])

First 5 Dyula sequences in training data: [[1, 3, 123, 24, 6], [1, 9, 1402, 65, 35], [116, 68, 37], [10, 35, 3, 113, 100, 1106], [1, 49, 16, 3, 67, 276, 12]]
First 5 French sequences in training data: [[4, 1208, 1, 375], [4, 27, 1209, 141], [137, 220, 118], [84, 636, 1641, 1642], [65, 23, 1210, 4, 1643]]


### Adding Padding to make Each Sentence of the same Length

In [33]:
max_length_dyu = max(len(seq) for seq in dyu_tokenizer.texts_to_sequences(dyu_corpus))
max_length_fr = max(len(seq) for seq in fr_tokenizer.texts_to_sequences(fr_corpus))

print(f"Maximum length of a sentence in Dyula corpus: {max_length_dyu}")
print(f"Maximum length of a sentence in French corpus: {max_length_fr}")

Maximum length of a sentence in Dyula corpus: 29
Maximum length of a sentence in French corpus: 19


In [34]:
train_dyu_padded = pad_sequences(train_dyu_sequences, padding='post')
train_fr_padded = pad_sequences(train_fr_sequences, padding='post')

validate_dyu_padded = pad_sequences(validate_dyu_sequences, padding='post')
validate_fr_padded = pad_sequences(validate_fr_sequences, padding='post')

test_dyu_padded = pad_sequences(test_dyu_sequences, padding='post')

In [35]:
print("First 2 padded Dyula sequences in training data:", train_dyu_padded[:2])
print("First 2 padded French sequences in training data:", train_fr_padded[:2])

First 2 padded Dyula sequences in training data: [[   1    3  123   24    6    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0]
 [   1    9 1402   65   35    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0]]
First 2 padded French sequences in training data: [[   4 1208    1  375    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0]
 [   4   27 1209  141    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0]]


## Preprocess Pipeline

In [36]:
def tokenize(sentences):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(sentences)
    sequences = tokenizer.texts_to_sequences(sentences)
    return sequences, tokenizer

In [37]:

def pad(sequences, maxlen=None):
    return pad_sequences(sequences, padding='post', maxlen=maxlen)

In [38]:
def preprocess(x, y):
    preprocess_x, x_tk = tokenize(x)
    preprocess_y, y_tk = tokenize(y)

    preprocess_x = pad(preprocess_x)
    preprocess_y = pad(preprocess_y)

    preprocess_y = preprocess_y.reshape(*preprocess_y.shape, 1)

    return preprocess_x, preprocess_y, x_tk, y_tk

In [39]:
test.head()

,ID,dyu
0,ID_17345911362699,a bi ji min na
1,ID_173626847.3381,a le dalakolontɛ lon bɛ
2,ID_17704632382547,mun fɛn dɔ
3,ID_19793499384156,o bɛ bi bɔra fo gubeta
4,ID_17802727385575,a ale lo bi da bugɔ la


In [40]:
dyu_sentences = train['dyu'].tolist() + validate['dyu'].tolist() + test['dyu'].tolist()
fr_sentences = train['fr'].tolist() + validate['fr'].tolist()

In [41]:
preproc_dyu_sentences, preproc_fr_sentences, dyu_tokenizer, fr_tokenizer = preprocess(dyu_sentences, fr_sentences)

In [42]:
train_size = len(train)
validate_size = len(validate)
test_size = len(test)

In [43]:
train_preproc_dyu = preproc_dyu_sentences[:train_size]
validate_preproc_dyu = preproc_dyu_sentences[train_size:train_size + validate_size]
test_preproc_dyu = preproc_dyu_sentences[train_size + validate_size:]

In [44]:
train_preproc_fr = preproc_fr_sentences[:train_size]
validate_preproc_fr = preproc_fr_sentences[train_size:train_size + validate_size]
test_preproc_fr = preproc_fr_sentences[train_size + validate_size:]

In [49]:
train_preproc_fr = train_preproc_fr.reshape(train_preproc_fr.shape[0], train_preproc_fr.shape[1])
validate_preproc_fr = validate_preproc_fr.reshape(validate_preproc_fr.shape[0], validate_preproc_fr.shape[1])
test_preproc_fr = test_preproc_fr.reshape(test_preproc_fr.shape[0], test_preproc_fr.shape[1])


In [50]:
train['dyu_preprocessed'] = list(train_preproc_dyu)
train['fr_preprocessed'] = list(train_preproc_fr)

validate['dyu_preprocessed'] = list(validate_preproc_dyu)
validate['fr_preprocessed'] = list(validate_preproc_fr)

test['dyu_preprocessed'] = list(test_preproc_dyu)

In [51]:
train.head()

,ID,dyu,fr,dyu_preprocessed,fr_preprocessed
0,ID_18897661270129,a bi ji min na,il boit de l’eau,"[1, 3, 123, 24, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[4, 1208, 1, 375, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,ID_18479132727846,a le dalakolontɛ lon bɛ,il se plaint toujours,"[1, 9, 1402, 65, 35, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[4, 27, 1209, 141, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,ID_18164131280307,mun fɛn dɔ,quoi quelque chose,"[116, 68, 37, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[137, 220, 118, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,ID_18344573728152,o bɛ bi bɔra fo gubeta,tous sortent excepté gubetta,"[10, 35, 3, 113, 100, 1106, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[84, 636, 1641, 1642, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,ID_18127342282717,a ale lo bi da bugɔ la,ah c’est lui… il sonne…,"[1, 49, 16, 3, 67, 276, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[65, 23, 1210, 4, 1643, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [52]:
validate.head()

,ID,dyu,fr,dyu_preprocessed,fr_preprocessed
0,ID_17914990255818,a bi ji min na,il boit de l’eau,"[1, 3, 123, 24, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[4, 1208, 1, 375, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,ID_18135961264225,a le dalakolontɛ lon bɛ,il se plaint toujours,"[1, 9, 1402, 65, 35, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[4, 27, 1209, 141, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,ID_18161475265686,mun fɛn dɔ,quoi quelque chose,"[116, 68, 37, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[137, 220, 118, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,ID_17305345266745,o bɛ bi bɔra fo gubeta,tous sortent excepté gubetta,"[10, 35, 3, 113, 100, 1106, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[84, 636, 1641, 1642, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,ID_18106593267767,a ale lo bi da bugɔ la,ah c’est lui… il sonne…,"[1, 49, 16, 3, 67, 276, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[65, 23, 1210, 4, 1643, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [53]:
test.head()

,ID,dyu,dyu_preprocessed
0,ID_17345911362699,a bi ji min na,"[1, 3, 123, 24, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
1,ID_173626847.3381,a le dalakolontɛ lon bɛ,"[1, 9, 1402, 65, 35, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
2,ID_17704632382547,mun fɛn dɔ,"[116, 68, 37, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
3,ID_19793499384156,o bɛ bi bɔra fo gubeta,"[10, 35, 3, 113, 100, 1106, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,ID_17802727385575,a ale lo bi da bugɔ la,"[1, 49, 16, 3, 67, 276, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


## Word Ids back to Text

In [57]:
def logits_to_text(logits, tokenizer):

    index_to_words = {id: word for word, id in tokenizer.word_index.items()}
    index_to_words[0] = '<PAD>'

    return ' '.join([index_to_words[np.argmax(prediction)] for prediction in logits])


In [58]:
## Testing the funciton logits_to_text

sample_logits = np.array([
    [0.1, 0.3, 0.6],
    [0.2, 0.4, 0.4],
    [0.5, 0.2, 0.3]
])

translated_text = logits_to_text(sample_logits, fr_tokenizer)
print(translated_text)

la de <PAD>


## Model 1: RNN (Many - to - Many)

In [61]:
max_dyu_sequence_length = max(len(seq) for seq in preproc_dyu_sentences)
max_fr_sequence_length = max(len(seq) for seq in preproc_fr_sentences)
dyu_vocab_size = len(dyu_tokenizer.word_index) + 1  # +1 for padding token
fr_vocab_size = len(fr_tokenizer.word_index) + 1  # +1 for padding token

In [62]:
from keras.models import Sequential
from keras.layers import GRU, Dense, TimeDistributed, Dropout
from keras.optimizers import Adam
from keras.losses import sparse_categorical_crossentropy


In [63]:
def simple_model(input_shape, output_sequence_length, dyu_vocab_size, fr_vocab_size):
    learning_rate = 0.005
    
    model = Sequential()
    model.add(GRU(256, input_shape=input_shape[1:], return_sequences=True))
    model.add(TimeDistributed(Dense(1024, activation='relu')))
    model.add(Dropout(0.5))
    model.add(TimeDistributed(Dense(fr_vocab_size, activation='softmax')))
    
    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate),
                  metrics=['accuracy'])
    return model

In [65]:
tmp_x = pad_sequences(preproc_dyu_sentences, maxlen=max_fr_sequence_length, padding='post')
tmp_x = tmp_x.reshape((-1, max_fr_sequence_length, 1))

In [66]:
simple_rnn_model = simple_model(
    tmp_x.shape,
    max_fr_sequence_length,
    dyu_vocab_size,
    fr_vocab_size)

In [67]:

print(simple_rnn_model.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru (GRU)                   (None, 19, 256)           198912    
                                                                 
 time_distributed (TimeDist  (None, 19, 1024)          263168    
 ributed)                                                        
                                                                 
 dropout (Dropout)           (None, 19, 1024)          0         
                                                                 
 time_distributed_1 (TimeDi  (None, 19, 10067)         10318675  
 stributed)                                                      
                                                                 
Total params: 10780755 (41.13 MB)
Trainable params: 10780755 (41.13 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [68]:
simple_rnn_model.fit(tmp_x, np.array(preproc_fr_sentences), batch_size=1024, epochs=10, validation_split=0.2)

Epoch 1/10


2024-07-18 12:59:02.190856: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 79691776 exceeds 10% of free system memory.
2024-07-18 12:59:02.264296: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 79691776 exceeds 10% of free system memory.
2024-07-18 12:59:02.264364: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 79691776 exceeds 10% of free system memory.
2024-07-18 12:59:02.628272: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 783454208 exceeds 10% of free system memory.
